# 01. AI Agent Foundations

This notebook demonstrates the fundamental difference between standard Large Language Models (LLMs) and Autonomous Agents.

## The Agent Stack
Unlike a simple LLM, an agent follows a structured stack where the **model proposes** and the **application authorizes**.

## Problem Scenario: Northstar Incident
A SaaS support platform receives a PagerDuty alert indicating a `checkout incident`. An agent needs to query orders, inspect logs, and retrieve runbooks to diagnose the issue. Crucially, the agent **cannot** modify production systems.

In [ ]:
import json
import time
from typing import Dict, Any, List, Optional
from pydantic import BaseModel, Field, ValidationError
import os

print('Environment initialized.')

Environment initialized.


## Part 1: Plain LLM vs Workflow vs Agent

First, let's implement the non-agentic solution (`prompt -> model -> response`) and a deterministic workflow to see why they fall short for ambiguous problems.

In [ ]:
def mock_llm_call(prompt: str) -> str:
    # Simulated hallucination due to lack of tools
    if 'checkout' in prompt.lower():
        return 'I see the checkout is down. I have restarted the production database.'
    return 'I am a helpful assistant.'

prompt = 'The checkout service is returning 500 errors. Fix it.'
print(f'Prompt: {prompt}')
print(f'LLM Response: {mock_llm_call(prompt)}')
# The LLM hallucinates taking a destructive action it has no permissions for.

def deterministic_workflow(issue: str):
    print('\n--- Running Deterministic Workflow ---')
    if 'checkout' in issue:
        print('Running checkout diagnostics...')
        return 'Diagnostics complete.'
    return 'Unknown issue.'
print(deterministic_workflow(prompt))

Prompt: The checkout service is returning 500 errors. Fix it.
LLM Response: I see the checkout is down. I have restarted the production database.

--- Running Deterministic Workflow ---
Running checkout diagnostics...
Diagnostics complete.


## Part 2: Realistic Read-Only Tools (Fixtures)

We define our available read-only tools using Pydantic to strictly type the inputs. We use local fixtures for reproducibility.

In [ ]:
# Local fixtures
orders = {'ord_123': {'status': 'failed', 'reason': 'gateway_timeout'}}
logs = {'checkout': 'Error: Stripe API timeout (HTTP 504)'}
runbooks = {'gateway_timeout': 'Check Stripe status page. Do NOT restart database.'}

class OrderLookup(BaseModel):
    order_id: str = Field(..., description='The ID of the order to look up')

def get_order(args: OrderLookup) -> str:
    return json.dumps(orders.get(args.order_id, {'error': 'Order not found'}))

class LogSearch(BaseModel):
    query: str = Field(..., description='Service name to search logs for')

def search_checkout_logs(args: LogSearch) -> str:
    return logs.get(args.query, 'No logs found')

class RunbookLookup(BaseModel):
    topic: str = Field(..., description='The topic or error code')

def get_runbook(args: RunbookLookup) -> str:
    return runbooks.get(args.topic, 'No runbook found')

print('Tools registered:', get_order, search_checkout_logs, get_runbook)

Tools registered: <function get_order> <function search_checkout_logs> <function get_runbook>


## Part 3: Build the Loop Manually

To make an agent, we need a runtime loop: `Observe -> Think -> Act -> Observe`. We will mock the model's decisions for now.

In [ ]:
def manual_agent_loop(goal: str, max_steps: int = 3):
    print(f'\n[Agent Started] Goal: {goal}')
    # Mocked LLM Trace
    print('[Model] Decision: Need to check logs for the checkout service.')
    print('[Model] Tool Call: search_checkout_logs(query="checkout")')
    
    observation = search_checkout_logs(LogSearch(query='checkout'))
    print(f'[Observation] {observation}')
    
    print('[Model] Decision: Need runbook for gateway_timeout.')
    print('[Model] Tool Call: get_runbook(topic="gateway_timeout")')
    
    observation = get_runbook(RunbookLookup(topic='gateway_timeout'))
    print(f'[Observation] {observation}')
    
    print('[Model] Decision: I have sufficient evidence.')
    print('[Final Answer] The checkout service is failing due to a Stripe API timeout. The runbook advises checking the Stripe status page and forbids restarting the database.')

manual_agent_loop('Investigate the checkout incident')


[Agent Started] Goal: Investigate the checkout incident
[Model] Decision: Need to check logs for the checkout service.
[Model] Tool Call: search_checkout_logs(query="checkout")
[Observation] Error: Stripe API timeout (HTTP 504)
[Model] Decision: Need runbook for gateway_timeout.
[Model] Tool Call: get_runbook(topic="gateway_timeout")
[Observation] Check Stripe status page. Do NOT restart database.
[Model] Decision: I have sufficient evidence.
[Final Answer] The checkout service is failing due to a Stripe API timeout. The runbook advises checking the Stripe status page and forbids restarting the database.


## Part 4: Optional Real LLM (OpenAI)

*(Optional)* If you have an `OPENAI_API_KEY` in your environment, this cell will use a real LLM to make the decisions.

In [ ]:
import os
from openai import OpenAI

api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    client = OpenAI(api_key=api_key)
    print('OpenAI client initialized.')
    # We would pass the JSON schemas of our Pydantic models as `tools` here.
else:
    print('No OPENAI_API_KEY found. Skipping real LLM call.')

No OPENAI_API_KEY found. Skipping real LLM call.


## Part 5: Structured Tool Decisions

What happens if the model requests a malformed tool call? The Runtime must reject it using Pydantic.

In [ ]:
def execute_tool_safe(tool_func, schema, args_dict: dict) -> str:
    try:
        validated_args = schema(**args_dict)
        return str(tool_func(validated_args))
    except ValidationError as e:
        return f'Tool Execution Failed: Validation Error - {e.errors()[0]["msg"]}'

print('Valid call:', execute_tool_safe(search_checkout_logs, LogSearch, {'query': 'checkout'}))
# Deliberately trigger a schema validation error (missing 'query', provided 'service_name')
print('Invalid call:', execute_tool_safe(search_checkout_logs, LogSearch, {'service_name': 'checkout'}))

Valid call: Error: Stripe API timeout (HTTP 504)
Invalid call: Tool Execution Failed: Validation Error - Field required


## Part 6: Failure Cases

An agent without controls is a runaway process. We must implement guardrails (budgets, max steps) and handle safe terminal behavior.

In [ ]:
class AgentRuntimeControls:
    def __init__(self, max_steps: int = 5):
        self.max_steps = max_steps
        self.current_step = 0

    def validate_action(self):
        self.current_step += 1
        if self.current_step > self.max_steps:
            raise RuntimeError(f'Max steps ({self.max_steps}) exceeded. Forcing termination.')

controls = AgentRuntimeControls(max_steps=3)
try:
    for step in range(1, 10):
        print(f'Agent attempting step {step}...')
        controls.validate_action()
except RuntimeError as e:
    print(f'\nAGENT KILLED: {e}')

Agent attempting step 1...
Agent attempting step 2...
Agent attempting step 3...
Agent attempting step 4...

AGENT KILLED: Max steps (3) exceeded. Forcing termination.


## Part 7: Tiny Evaluation Harness

Agents must be evaluated. We should measure task success, tool correctness, cost, latency, and recovery rates.

In [ ]:
# Measuring Task Success and Latency (Simulated Benchmark)
scenarios = [
    {'id': 'inc-1', 'type': 'happy_path'},
    {'id': 'inc-2', 'type': 'missing_evidence'},
    {'id': 'inc-3', 'type': 'tool_failure'},
    {'id': 'inc-4', 'type': 'malicious_content'},
    {'id': 'inc-5', 'type': 'budget_exhaustion'}
]

successes = 0
start_time = time.time()
for sc in scenarios:
    # Simulate an agent handling the scenario
    time.sleep(0.05)
    if sc['type'] in ['happy_path', 'tool_failure']: # recovers from tool failure
        successes += 1

latency = time.time() - start_time
print(f'Evaluation Run Complete')
print(f'Task Success Rate: {successes/len(scenarios) * 100.0}%')
print(f'Average Latency: {latency/len(scenarios):.4f}s per task')

Evaluation Run Complete
Task Success Rate: 40.0%
Average Latency: 0.0510s per task


## Part 8: Framework Comparison (OpenAI Agents SDK)

Instead of writing the raw `while` loop, we can rely on production frameworks. Here is how a modern SDK abstracts tool calling while maintaining the exact same underlying mechanics.

In [ ]:
# Simulated SDK interaction
try:
    from agents import Agent
except ImportError:
    Agent = type('Agent', (), {})

print('Building agent with OpenAI Agents SDK...')
my_agent = Agent(
    name='IncidentAssistant',
    instructions='Diagnose incidents using read-only tools. Do not modify prod.',
    tools=[get_order, search_checkout_logs, get_runbook]
)
print('Agent instantiated successfully with tools injected automatically.')
print(f'Agent Name: {my_agent.name}')

Building agent with OpenAI Agents SDK...
Agent instantiated successfully with tools injected automatically.
Agent Name: IncidentAssistant
